In [ ]:
# pandas: đọc và xử lý bảng dữ liệu; numpy: cung cấp giá trị NaN (ô khuyết)
import os
import pandas as pd
import numpy as np

# Cho phép hiển thị đầy đủ tất cả các cột khi in bảng
pd.set_option("display.max_columns", None)

print("Phiên bản pandas:", pd.__version__)

In [ ]:
# Nạp thư viện os để kiểm tra file có tồn tại hay không
import os

# Tên file mặc định
FILE = "adult.csv"

# Nếu chưa có file trong thư mục làm việc thì hiện hộp thoại upload (chỉ chạy được trên Colab)
if not os.path.exists(FILE):
    try:
        from google.colab import files        # thư viện riêng của Colab
        uploaded = files.upload()             # mở hộp thoại chọn file từ máy tính
        FILE = list(uploaded.keys())[0]       # lấy tên file vừa upload (có thể là 'adult (1).csv')
    except ImportError:
        raise FileNotFoundError("Không tìm thấy adult.csv. Hãy đặt file cùng thư mục với notebook.")

print("Sẽ đọc file:", FILE)

In [ ]:
# skipinitialspace=True: bỏ khoảng trắng thừa ở đầu mỗi giá trị chuỗi
df = pd.read_csv(FILE, skipinitialspace=True)
print("Đã đọc file thành công")

In [ ]:
# df.shape trả về (số dòng, số cột)
print("Kích thước (dòng, cột):", df.shape)
print("Số quan sát (dòng)   :", df.shape[0])
print("Số thuộc tính (cột)  :", df.shape[1])

In [ ]:
# info(): tên cột, số giá trị không-null và kiểu dữ liệu từng cột
# Lúc này giá trị khuyết còn là ký tự '?' nên info() chưa báo thiếu
df.info()

In [ ]:
df.head()

In [ ]:
# count, mean, std, min, 25%, median (50%), 75%, max của các cột số
df.describe()

In [ ]:
# Trước khi đổi: isna() báo 0 vì '?' chỉ là một chuỗi bình thường
print("Số NaN trước khi đổi:", df.isna().sum().sum())

# Đổi '?' thành NaN và GÁN lại vào df để các bước sau dùng dữ liệu đã đổi
df = df.replace("?", np.nan)

# Đếm số ô khuyết của từng cột
print("\nSố ô khuyết theo cột:")
print(df.isna().sum())

In [ ]:
missing = pd.DataFrame({
    "so_o_khuyet": df.isna().sum(),
    "ty_le_%": (df.isna().mean() * 100).round(2),
    "kieu_du_lieu": df.dtypes.astype(str),
}).sort_values("so_o_khuyet", ascending=False)

missing

In [ ]:
so_dong_khuyet = int(df.isna().any(axis=1).sum())
print("Số cột có khuyết       :", int((df.isna().sum() > 0).sum()), "/", df.shape[1])
print("Tổng số ô khuyết       :", int(df.isna().sum().sum()))
print("Số dòng có >= 1 ô khuyết:", so_dong_khuyet, f"({so_dong_khuyet / len(df) * 100:.2f}%)")

In [ ]:
so_trung = int(df.duplicated().sum())
print("Số dòng trùng lặp:", so_trung, f"({so_trung / len(df) * 100:.2f}%)")

In [ ]:
print(df["income"].value_counts(normalize=True))

pd.DataFrame({
    "so_luong": df["income"].value_counts(),
    "ty_le_%": (df["income"].value_counts(normalize=True) * 100).round(2),
})

In [ ]:
# Kỳ vọng: Husband gần như toàn Male, Wife gần như toàn Female
pd.crosstab(df["relationship"], df["sex"])

In [ ]:
so_cot_so = df.select_dtypes(include="number").shape[1]
print(f"- Quy mô: {df.shape[0]:,} dòng x {df.shape[1]} cột ({so_cot_so} cột số, {df.shape[1] - so_cot_so} cột chuỗi).")
for ten_cot, hang in missing[missing["so_o_khuyet"] > 0].iterrows():
    print(f"- Cột '{ten_cot}' khuyết {int(hang['so_o_khuyet']):,} ô ({hang['ty_le_%']}%).")
print(f"- {so_dong_khuyet:,} dòng có ít nhất một ô khuyết.")
print(f"- Dòng trùng lặp: {so_trung}.")
print("\nHOÀN TẤT: notebook đã chạy xong không lỗi.")

In [ ]:
import matplotlib.pyplot as plt                       # vẽ biểu đồ cơ bản
import seaborn as sns                                 # vẽ heatmap, boxplot, histogram
from IPython.display import display                   # hiển thị nhiều bảng trong 1 cell
from sklearn.model_selection import train_test_split  # chia tập train/test
from sklearn.preprocessing import StandardScaler      # chuẩn hóa z-score

sns.set_theme(style="whitegrid")

df_before = df.copy()   # lưu bản trước khi làm sạch để so sánh ở cuối

# Đổi dấu chấm trong tên cột thành gạch dưới ("education.num" -> "education_num")
df.columns = df.columns.str.replace(".", "_", regex=False)
print("Tên cột sau khi đổi:", df.columns.tolist())

In [ ]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()   # các cột chữ

audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),                   # kiểu dữ liệu
    "so_gia_tri_khac_nhau": df.nunique(),             # số giá trị phân biệt
    "so_o_thieu": df.isna().sum(),                    # số ô NaN
    "khoang_trang_thua": [                            # ô có khoảng trắng thừa (chỉ cột chữ)
        (df[c].dropna() != df[c].dropna().str.strip()).sum() if c in cat_cols else 0
        for c in df.columns
    ],
})
display(audit)
print("\nCác giá trị của income:", df["income"].unique())   # phải đúng 2 nhãn

In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(df.isna(), cbar=False, cmap="viridis", yticklabels=False)
plt.title("Bản đồ nhiệt các ô thiếu (vạch sáng = thiếu)")
plt.show()

In [ ]:
# workclass và occupation có thiếu cùng nhau không?
display(pd.crosstab(df["workclass"].isna(), df["occupation"].isna(),
                    rownames=["workclass thiếu?"], colnames=["occupation thiếu?"]))

# Tỷ lệ thu nhập >50K: nhóm có ô thiếu so với nhóm đủ thông tin
co_o_thieu = df.isna().any(axis=1)
ty_le_50k = df.groupby(co_o_thieu)["income"].apply(lambda s: (s == ">50K").mean())
ty_le_50k = ty_le_50k.rename({False: "Đủ thông tin", True: "Có ô thiếu"})
print("Tỷ lệ thu nhập >50K theo nhóm:")
print((ty_le_50k * 100).round(1).astype(str) + " %")

In [ ]:
num_cols = ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
so_sanh = pd.DataFrame({
    "mean": df[num_cols].mean(),
    "median": df[num_cols].median(),
    "skew": df[num_cols].skew(),                      # skew > 1: lệch phải mạnh
}).round(2)
so_sanh["mean_tru_median"] = (so_sanh["mean"] - so_sanh["median"]).round(2)
display(so_sanh)

In [ ]:
for col in ["workclass", "occupation"]:
    df[col] = df[col].fillna("Unknown")               # nhãn riêng cho nhóm không khai báo

mode_country = df["native_country"].mode()[0]         # giá trị xuất hiện nhiều nhất
df["native_country"] = df["native_country"].fillna(mode_country)

print("Mode của native_country:", mode_country)
print("Tổng số ô thiếu còn lại:", df.isna().sum().sum())   # kỳ vọng = 0

In [ ]:
so_trung = df.duplicated().sum()
print("Số dòng trùng:", so_trung)

# keep=False: hiện cả bản gốc lẫn bản sao, sắp xếp để các cặp nằm cạnh nhau
dong_trung = df[df.duplicated(keep=False)].sort_values(list(df.columns))
display(dong_trung.head(6))

# Đối chứng: nếu bỏ fnlwgt thì số dòng trùng là bao nhiêu?
print("Số giá trị fnlwgt khác nhau:", df["fnlwgt"].nunique(), "/", len(df), "dòng")
print("Số dòng trùng nếu bỏ fnlwgt :", df.drop(columns="fnlwgt").duplicated().sum())
print("Số dòng trùng nếu giữ fnlwgt:", so_trung)

In [ ]:
truoc = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)   # gán lại, không dùng inplace
print(f"Đã xóa {truoc - len(df)} dòng. Còn lại {len(df)} dòng.")

In [ ]:
def iqr_bounds(s):
    q1 = s.quantile(0.25)                              # tứ phân vị thứ nhất
    q3 = s.quantile(0.75)                              # tứ phân vị thứ ba
    iqr = q3 - q1                                      # khoảng tứ phân vị
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr              # (ngưỡng dưới, ngưỡng trên)

lower_bound, upper_bound = iqr_bounds(df["age"])
outliers = df[(df["age"] < lower_bound) | (df["age"] > upper_bound)]
print(f"age: ngưỡng [{lower_bound:.1f}, {upper_bound:.1f}] -> {len(outliers)} ngoại lệ ({len(outliers)/len(df):.2%})")

In [ ]:
dong = []
for c in num_cols:
    lo, hi = iqr_bounds(df[c])
    so_ngoai_le = int(((df[c] < lo) | (df[c] > hi)).sum())
    dong.append({"cot": c, "min": df[c].min(), "max": df[c].max(),
                 "nguong_duoi": lo, "nguong_tren": hi,
                 "so_ngoai_le": so_ngoai_le, "ty_le_%": 100 * so_ngoai_le / len(df),
                 "skew": df[c].skew()})
bang_iqr = pd.DataFrame(dong).set_index("cot").round(2)
display(bang_iqr)

In [ ]:
fig, axes = plt.subplots(2, len(num_cols), figsize=(22, 7))
for i, c in enumerate(num_cols):
    sns.boxplot(y=df[c], ax=axes[0, i], color="lightsteelblue")
    axes[0, i].set_title(f"{c}\nskew = {df[c].skew():.2f}")
    sns.histplot(df[c], bins=40, ax=axes[1, i], color="steelblue")
plt.tight_layout()
plt.show()

In [ ]:
cg = df["capital_gain"]
nhom_99999 = df[cg == 99999]
print("capital_gain = 0     :", f"{(cg == 0).mean():.1%} số dòng")
print("capital_gain = 99999 :", len(nhom_99999), "dòng; tỷ lệ >50K:", f"{(nhom_99999['income'] == '>50K').mean():.0%}")
print("hours_per_week = 40  :", f"{(df['hours_per_week'] == 40).mean():.1%} số dòng")
print("hours_per_week >= 80 :", (df["hours_per_week"] >= 80).sum(), "dòng")
print("age = 90             :", (df["age"] == 90).sum(), "dòng")

In [ ]:
df["capital_gain_capped"] = (df["capital_gain"] == 99999).astype(int)   # 1 nếu bị chặn trên
df["has_capital_gain"] = (df["capital_gain"] > 0).astype(int)
df["has_capital_loss"] = (df["capital_loss"] > 0).astype(int)
for c in ["fnlwgt", "capital_gain", "capital_loss"]:
    df[c + "_log"] = np.log1p(df[c])                  # log(1 + x), dùng được khi x = 0

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for i, c in enumerate(["fnlwgt", "capital_gain", "capital_loss"]):
    sns.histplot(df[c], bins=40, ax=axes[0, i], color="tab:red")
    axes[0, i].set_title(f"{c} (trước) skew = {df[c].skew():.2f}")
    sns.histplot(df[c + "_log"], bins=40, ax=axes[1, i], color="tab:green")
    axes[1, i].set_title(f"{c}_log (sau) skew = {df[c + '_log'].skew():.2f}")
plt.tight_layout()
plt.show()

In [ ]:
print("education -> education_num tối đa", df.groupby("education")["education_num"].nunique().max(), "giá trị")
print("education_num -> education tối đa", df.groupby("education_num")["education"].nunique().max(), "giá trị")
# Cả hai = 1 nghĩa là hai cột chứa cùng thông tin -> bỏ cột chữ, giữ cột số
df = df.drop(columns="education")

df.to_csv("adult_clean.csv", index=False)
print("Kích thước bản sạch:", df.shape)

In [ ]:
df_enc = df.copy()
df_enc["income"] = (df_enc["income"] == ">50K").astype(int)          # 1 = trên 50K
df_enc["sex"] = (df_enc["sex"] == "Male").astype(int)                # 1 = Male
df_enc["is_us"] = (df_enc["native_country"] == "United-States").astype(int)

nominal = ["workclass", "marital_status", "occupation", "relationship", "race"]
df_enc = pd.get_dummies(df_enc, columns=nominal, drop_first=True, dtype=int)

df_enc = df_enc.drop(columns=["native_country", "fnlwgt", "capital_gain", "capital_loss"])
print("Kích thước sau mã hóa:", df_enc.shape)
print("Còn cột chữ nào không? ->", df_enc.select_dtypes(exclude="number").columns.tolist())   # kỳ vọng []

In [ ]:
X = df_enc.drop(columns="income")                      # đặc trưng
y = df_enc["income"]                                   # nhãn

# stratify=y giữ tỷ lệ >50K ở cả hai tập; random_state để kết quả lặp lại được
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

cols_scale = ["age", "education_num", "hours_per_week", "fnlwgt_log", "capital_gain_log", "capital_loss_log"]
scaler = StandardScaler().fit(X_train[cols_scale])     # chỉ học mean/std từ tập train

X_train_s, X_test_s = X_train.copy(), X_test.copy()
X_train_s[cols_scale] = scaler.transform(X_train[cols_scale])
X_test_s[cols_scale] = scaler.transform(X_test[cols_scale])

print("Tỷ lệ >50K: train =", round(y_train.mean(), 4), "| test =", round(y_test.mean(), 4))
display(X_train_s[cols_scale].describe().loc[["mean", "std"]].round(3))   # kỳ vọng mean ~ 0, std ~ 1

In [ ]:
tong_ket = pd.DataFrame({
    "Trước": [df_before.shape[0], int(df_before.isna().sum().sum()), int(df_before.duplicated().sum()),
              round(df_before["capital.gain"].skew(), 2)],
    "Sau": [df.shape[0], int(df.isna().sum().sum()), int(df.duplicated().sum()),
            round(df["capital_gain_log"].skew(), 2)],
}, index=["Số dòng", "Số ô thiếu", "Số dòng trùng", "Skew của capital_gain (Sau = bản log1p)"])
display(tong_ket)

df_enc.to_csv("adult_encoded.csv", index=False)
print("Đã lưu adult_clean.csv và adult_encoded.csv")

In [ ]:
import os, math, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams.update({"axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 11})

# Nếu chạy riêng Giai đoạn 3 (chưa có biến df từ Giai đoạn 2) thì nạp file adult_clean.csv
if "df" not in globals():
    FILE_CLEAN = "adult_clean.csv"
    if not os.path.exists(FILE_CLEAN):
        from google.colab import files
        up = files.upload()
        FILE_CLEAN = list(up.keys())[0]
    df = pd.read_csv(FILE_CLEAN)

# Đảm bảo tên cột đã được chuẩn hóa (đổi dấu chấm thành gạch dưới nếu chưa đổi)
df.columns = df.columns.str.replace(".", "_", regex=False)

# Làm việc trên bản sao "dfa" để không làm thay đổi df của Giai đoạn 2
dfa = df.copy()
dfa["income_bin"] = (dfa["income"] == ">50K").astype(int)
BASE = dfa["income_bin"].mean()
print(f"Số quan sát: {len(dfa):,} | Tỷ lệ nền >50K: {BASE:.2%}")

PALETTE = {"<=50K": "#6c8ebf", ">50K": "#d9534f"}
C_LOW, C_HIGH = "#6c8ebf", "#d9534f"

OUT_DIR = "hinh_giai_doan3"
os.makedirs(OUT_DIR, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(f"{OUT_DIR}/{name}.png", dpi=150, bbox_inches="tight")

# Bảng đổi education_num -> tên bậc học (cột 'education' đã bị xóa ở Giai đoạn 2)
EDU_NAMES = {1: "Preschool", 2: "1st-4th", 3: "5th-6th", 4: "7th-8th", 5: "9th", 6: "10th",
             7: "11th", 8: "12th", 9: "HS-grad", 10: "Some-college", 11: "Assoc-voc",
             12: "Assoc-acdm", 13: "Bachelors", 14: "Masters", 15: "Prof-school", 16: "Doctorate"}

dfa["age_group"] = pd.cut(dfa["age"], bins=[17, 25, 35, 45, 55, 65, 100], right=False,
                          labels=["17-24", "25-34", "35-44", "45-54", "55-64", "65+"])
dfa["hours_grp"] = pd.cut(dfa["hours_per_week"], bins=[0, 20, 39, 40, 50, 60, 99],
                          labels=["<=20", "21-39", "40", "41-50", "51-60", ">60"])
dfa["edu_grp"] = pd.cut(dfa["education_num"], bins=[0, 8, 9, 12, 13, 16],
                        labels=["Dưới THPT", "HS-grad", "Some-college/Assoc", "Bachelors", "Sau đại học"])
dfa["is_married"] = (dfa["marital_status"] == "Married-civ-spouse").astype(int)
dfa["is_male"] = (dfa["sex"] == "Male").astype(int)

def rate_table(col):
    t = dfa.groupby(col, observed=True)["income_bin"].agg(n="size", n_high="sum", rate="mean")
    t["rate_%"] = (t["rate"] * 100).round(1)
    return t

def cramers_v(ct):
    chi2 = chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum()
    return math.sqrt(chi2 / (n * (min(ct.shape) - 1)))

def chi2_test(series):
    ct = pd.crosstab(series, dfa["income_bin"])
    chi2, p, dof, _ = chi2_contingency(ct)
    return chi2, p, dof, cramers_v(ct)

def mw_test(col):
    # AUC = xác suất một người >50K có giá trị cao hơn một người <=50K (0.5 = không khác biệt)
    hi = dfa.loc[dfa["income_bin"] == 1, col]
    lo = dfa.loc[dfa["income_bin"] == 0, col]
    u, p = mannwhitneyu(hi, lo, alternative="two-sided")
    return u, p, u / (len(hi) * len(lo))

def fmt_p(p):
    return "<1e-300" if p < 1e-300 else f"{p:.2e}"

def bar_text(ax, bars, texts, horizontal=False):
    for b, t in zip(bars, texts):
        if horizontal:
            ax.text(b.get_width() + 0.8, b.get_y() + b.get_height() / 2, t, va="center", fontsize=9)
        else:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.8, t, ha="center", fontsize=9)

ket_qua = []   # gom kết quả kiểm định để lập bảng tổng kết ở cuối
print("Đã thiết lập xong.")

In [ ]:
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams.update({"axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 11})

# Chưa có df (chưa chạy Giai đoạn 1-2) thì tự tải bộ dữ liệu từ UCI, không cần upload
if "df" not in globals():
    cols = "age workclass fnlwgt education education.num marital.status occupation relationship race sex capital.gain capital.loss hours.per.week native.country income".split()
    df = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data", header=None, names=cols, skipinitialspace=True)

# Làm sạch lại theo đúng quy trình Giai đoạn 2 của nhóm (chạy lại nhiều lần vẫn cho cùng kết quả)
dfa = df.copy()
dfa.columns = dfa.columns.str.replace(".", "_", regex=False)
dfa = dfa.replace("?", np.nan)
dfa[["workclass", "occupation"]] = dfa[["workclass", "occupation"]].fillna("Unknown")
dfa["native_country"] = dfa["native_country"].fillna(dfa["native_country"].mode()[0])
dfa = dfa.drop_duplicates().reset_index(drop=True)
for c in ["capital_gain", "capital_loss"]:
    dfa[c + "_log"] = np.log1p(dfa[c])
dfa = dfa.drop(columns="education", errors="ignore")

dfa["income_bin"] = (dfa["income"] == ">50K").astype(int)
BASE = dfa["income_bin"].mean()
print(f"Số quan sát: {len(dfa):,} | Tỷ lệ nền >50K: {BASE:.2%}")
print("Kỳ vọng: 32,537 dòng và 24.09%" if len(dfa) != 32537 else "Khớp với Giai đoạn 2 của nhóm (32,537 dòng).")

PALETTE = {"<=50K": "#6c8ebf", ">50K": "#d9534f"}
C_LOW, C_HIGH = "#6c8ebf", "#d9534f"

OUT_DIR = "hinh_giai_doan3"
os.makedirs(OUT_DIR, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(f"{OUT_DIR}/{name}.png", dpi=150, bbox_inches="tight")

# Bảng đổi education_num -> tên bậc học (cột 'education' đã bị xóa ở Giai đoạn 2)
EDU_NAMES = {1: "Preschool", 2: "1st-4th", 3: "5th-6th", 4: "7th-8th", 5: "9th", 6: "10th",
             7: "11th", 8: "12th", 9: "HS-grad", 10: "Some-college", 11: "Assoc-voc",
             12: "Assoc-acdm", 13: "Bachelors", 14: "Masters", 15: "Prof-school", 16: "Doctorate"}

dfa["age_group"] = pd.cut(dfa["age"], bins=[17, 25, 35, 45, 55, 65, 100], right=False,
                          labels=["17-24", "25-34", "35-44", "45-54", "55-64", "65+"])
dfa["hours_grp"] = pd.cut(dfa["hours_per_week"], bins=[0, 20, 39, 40, 50, 60, 99],
                          labels=["<=20", "21-39", "40", "41-50", "51-60", ">60"])
dfa["edu_grp"] = pd.cut(dfa["education_num"], bins=[0, 8, 9, 12, 13, 16],
                        labels=["Dưới THPT", "HS-grad", "Some-college/Assoc", "Bachelors", "Sau đại học"])
dfa["is_married"] = (dfa["marital_status"] == "Married-civ-spouse").astype(int)
dfa["is_male"] = (dfa["sex"] == "Male").astype(int)

def rate_table(col):
    t = dfa.groupby(col, observed=True)["income_bin"].agg(n="size", n_high="sum", rate="mean")
    t["rate_%"] = (t["rate"] * 100).round(1)
    return t

def cramers_v(ct):
    chi2 = chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum()
    return math.sqrt(chi2 / (n * (min(ct.shape) - 1)))

def chi2_test(series):
    ct = pd.crosstab(series, dfa["income_bin"])
    chi2, p, dof, _ = chi2_contingency(ct)
    return chi2, p, dof, cramers_v(ct)

def mw_test(col):
    # AUC = xác suất một người >50K có giá trị cao hơn một người <=50K (0.5 = không khác biệt)
    hi = dfa.loc[dfa["income_bin"] == 1, col]
    lo = dfa.loc[dfa["income_bin"] == 0, col]
    u, p = mannwhitneyu(hi, lo, alternative="two-sided")
    return u, p, u / (len(hi) * len(lo))

def fmt_p(p):
    return "<1e-300" if p < 1e-300 else f"{p:.2e}"

def bar_text(ax, bars, texts, horizontal=False):
    for b, t in zip(bars, texts):
        if horizontal:
            ax.text(b.get_width() + 0.8, b.get_y() + b.get_height() / 2, t, va="center", fontsize=9)
        else:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.8, t, ha="center", fontsize=9)

ket_qua = []   # gom kết quả kiểm định để lập bảng tổng kết ở cuối
print("Đã thiết lập xong.")

In [ ]:
cnt = dfa["income"].value_counts().reindex(["<=50K", ">50K"])
fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(cnt.index, cnt.values, color=[C_LOW, C_HIGH])
for b, v in zip(bars, cnt.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 300, f"{v:,}\n({v / len(dfa):.1%})", ha="center", fontsize=10)
ax.set_ylim(0, cnt.max() * 1.18)
ax.set_title("Hình 1. Phân bố nhóm thu nhập")
ax.set_ylabel("Số người")
plt.tight_layout()
save_fig(fig, "hinh1_tong_quan_income")
plt.show()

print("SỐ LIỆU CHÍNH (dùng để viết Deep Insight):")
display(pd.DataFrame({"so_luong": cnt, "ty_le_%": (cnt / len(dfa) * 100).round(2)}))
print(f"- Tỷ lệ <=50K / >50K = {cnt['<=50K'] / cnt['>50K']:.2f} : 1")

In [ ]:
t_age = rate_table("age_group")
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(data=dfa, x="age", hue="income", hue_order=["<=50K", ">50K"], palette=PALETTE,
             stat="density", common_norm=False, element="step", bins=range(17, 93, 2), ax=axes[0])
for inc, color in PALETTE.items():
    axes[0].axvline(dfa.loc[dfa["income"] == inc, "age"].median(), color=color, ls=":", lw=2)
axes[0].set_title("(a) Phân phối tuổi theo nhóm thu nhập (đường chấm = trung vị)")
axes[0].set_xlabel("Tuổi")
axes[0].set_ylabel("Mật độ")

labels = t_age.index.astype(str).tolist()
axes[1].plot(labels, t_age["rate_%"], marker="o", lw=2.5, color=C_HIGH)
for x, y, n in zip(labels, t_age["rate_%"], t_age["n"]):
    axes[1].annotate(f"{y:.1f}%\n(n={n:,})", (x, y), textcoords="offset points", xytext=(0, 10),
                     ha="center", fontsize=9)
axes[1].axhline(BASE * 100, color="gray", ls="--", label=f"Tỷ lệ nền {BASE:.1%}")
axes[1].legend(loc="upper right")
axes[1].set_ylim(0, t_age["rate_%"].max() + 15)
axes[1].set_title("(b) Tỷ lệ >50K theo nhóm tuổi")
axes[1].set_xlabel("Nhóm tuổi")
axes[1].set_ylabel("% người thu nhập >50K")
plt.suptitle("Hình 2. Tuổi tác và thu nhập", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_fig(fig, "hinh2_tuoi")
plt.show()

u, p, auc = mw_test("age")
med_hi = dfa.loc[dfa["income_bin"] == 1, "age"].median()
med_lo = dfa.loc[dfa["income_bin"] == 0, "age"].median()
peak = t_age["rate"].idxmax()
idx_peak = t_age.index.tolist().index(peak)
supported = 0 < idx_peak < len(t_age) - 1 and p < 0.05
ket_qua.append({"Giả thuyết": "H1 Tuổi: tăng rồi giảm (U ngược)", "Kiểm định": "Mann-Whitney U",
                "p-value": fmt_p(p), "Độ lớn hiệu ứng": f"AUC = {auc:.3f}",
                "Kết luận": "Ủng hộ" if supported else "Không ủng hộ"})

print("SỐ LIỆU CHÍNH (dùng để viết Deep Insight):")
display(t_age[["n", "rate_%"]])
print(f"- Tuổi trung vị: nhóm >50K = {med_hi:.0f}, nhóm <=50K = {med_lo:.0f} (chênh {med_hi - med_lo:.0f} tuổi)")
print(f"- Nhóm tuổi đạt đỉnh: {peak} với {t_age.loc[peak, 'rate']:.1%} (gấp {t_age.loc[peak, 'rate'] / BASE:.2f} lần tỷ lệ nền)")
print(f"- Nhóm thấp nhất: {t_age['rate'].idxmin()} với {t_age['rate'].min():.1%}")
print(f"- Mann-Whitney U: p = {fmt_p(p)}, AUC = {auc:.3f} (xác suất người >50K lớn tuổi hơn người <=50K)")
print(f"- H1 (hình chữ U ngược): {'ĐƯỢC ỦNG HỘ' if supported else 'KHÔNG ĐƯỢC ỦNG HỘ'}")

In [ ]:
t_edu = rate_table("education_num").sort_index()
names = [EDU_NAMES.get(i, str(i)) for i in t_edu.index]
colors = [C_HIGH if r > BASE else C_LOW for r in t_edu["rate"]]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(names, t_edu["rate_%"], color=colors)
bar_text(ax, bars, [f"{r:.1f}%  (n={n:,})" for r, n in zip(t_edu["rate_%"], t_edu["n"])], horizontal=True)
ax.axvline(BASE * 100, color="gray", ls="--")
ax.text(BASE * 100 + 0.5, -0.9, f"Tỷ lệ nền {BASE:.1%}", color="gray", fontsize=9)
ax.set_xlim(0, t_edu["rate_%"].max() + 20)
ax.set_title("Hình 3. Tỷ lệ thu nhập >50K theo bậc học (thấp -> cao)")
ax.set_xlabel("% người thu nhập >50K")
ax.set_ylabel("Bậc học")
plt.tight_layout()
save_fig(fig, "hinh3_hoc_van")
plt.show()

rho, p_rho = spearmanr(dfa["education_num"], dfa["income_bin"])
rho_grp, _ = spearmanr(t_edu.index, t_edu["rate"])
chi2, p, dof, v = chi2_test(dfa["education_num"])
supported = rho > 0 and rho_grp > 0.9 and p < 0.05
ket_qua.append({"Giả thuyết": "H2 Học vấn càng cao, >50K càng cao", "Kiểm định": "Chi-square + Spearman",
                "p-value": fmt_p(p), "Độ lớn hiệu ứng": f"V = {v:.3f}; rho = {rho:.3f}",
                "Kết luận": "Ủng hộ" if supported else "Không ủng hộ"})

def r_edu(k):
    return t_edu.loc[k, "rate"] if k in t_edu.index else float("nan")

first_above = next((EDU_NAMES[i] for i in t_edu.index if t_edu.loc[i, "rate"] > BASE), "không có")
print("SỐ LIỆU CHÍNH (dùng để viết Deep Insight):")
print(f"- HS-grad (9): {r_edu(9):.1%} | Some-college (10): {r_edu(10):.1%} | Bachelors (13): {r_edu(13):.1%} | "
      f"Masters (14): {r_edu(14):.1%} | Prof-school (15): {r_edu(15):.1%} | Doctorate (16): {r_edu(16):.1%}")
print(f"- Bachelors gấp HS-grad {r_edu(13) / r_edu(9):.2f} lần; Doctorate gấp HS-grad {r_edu(16) / r_edu(9):.2f} lần")
print(f"- Bậc học đầu tiên vượt tỷ lệ nền: {first_above}")
print(f"- Spearman (cá nhân) rho = {rho:.3f}, p = {fmt_p(p_rho)} | Spearman (theo nhóm) rho = {rho_grp:.3f}")
print(f"- Chi-square: chi2 = {chi2:,.0f}, dof = {dof}, p = {fmt_p(p)}, Cramér's V = {v:.3f}")
print(f"- H2: {'ĐƯỢC ỦNG HỘ' if supported else 'KHÔNG ĐƯỢC ỦNG HỘ'}")

In [ ]:
t_sex = rate_table("sex").reindex(["Female", "Male"])
dfa["marital_grp"] = np.where(dfa["is_married"] == 1, "Đang kết hôn (civ-spouse)", "Nhóm còn lại")
t_sm = dfa.groupby(["marital_grp", "sex"], observed=True)["income_bin"].agg(n="size", rate="mean").reset_index()
t_sm["rate_%"] = t_sm["rate"] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars = axes[0].bar(t_sex.index, t_sex["rate_%"], color=["#e8a0bf", "#6c8ebf"])
bar_text(axes[0], bars, [f"{r:.1f}%\n(n={n:,})" for r, n in zip(t_sex["rate_%"], t_sex["n"])])
axes[0].axhline(BASE * 100, color="gray", ls="--")
axes[0].set_ylim(0, t_sex["rate_%"].max() + 10)
axes[0].set_title("(a) Tỷ lệ >50K theo giới tính")
axes[0].set_ylabel("% người thu nhập >50K")
sns.barplot(data=t_sm, x="marital_grp", y="rate_%", hue="sex", hue_order=["Female", "Male"],
            palette={"Female": "#e8a0bf", "Male": "#6c8ebf"}, ax=axes[1])
for c in axes[1].containers:
    axes[1].bar_label(c, fmt="%.1f%%", padding=2, fontsize=9)
axes[1].set_ylim(0, t_sm["rate_%"].max() + 10)
axes[1].set_title("(b) Tỷ lệ >50K theo giới tính, tách theo tình trạng hôn nhân")
axes[1].set_xlabel("")
axes[1].set_ylabel("% người thu nhập >50K")
plt.suptitle("Hình 4. Giới tính và thu nhập", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_fig(fig, "hinh4_gioi_tinh")
plt.show()

chi2, p, dof, v = chi2_test(dfa["sex"])
rate_f, rate_m = t_sex.loc["Female", "rate"], t_sex.loc["Male", "rate"]
supported = rate_m > rate_f and p < 0.05
ket_qua.append({"Giả thuyết": "H3 Nam có >50K cao hơn nữ", "Kiểm định": "Chi-square",
                "p-value": fmt_p(p), "Độ lớn hiệu ứng": f"V = {v:.3f}",
                "Kết luận": "Ủng hộ" if supported else "Không ủng hộ"})

print("SỐ LIỆU CHÍNH (dùng để viết Deep Insight):")
print(f"- Nam: {rate_m:.1%} | Nữ: {rate_f:.1%} | Chênh {100 * (rate_m - rate_f):.1f} điểm % | Nam gấp nữ {rate_m / rate_f:.2f} lần")
print(f"- Chi-square: chi2 = {chi2:,.0f}, p = {fmt_p(p)}, Cramér's V = {v:.3f}")
print("\nTỷ lệ đang kết hôn (civ-spouse) theo giới (yếu tố gây nhiễu):")
print((dfa.groupby("sex")["is_married"].mean().mul(100).round(1)).to_string())
print("\nTỷ lệ >50K theo giới, tách theo hôn nhân (%):")
display((t_sm.pivot(index="marital_grp", columns="sex", values="rate_%")).round(1))
gap_tbl = dfa.groupby(["hours_grp", "sex"], observed=True)["income_bin"].mean().unstack().mul(100).round(1)
gap_tbl["Chênh Nam-Nữ"] = (gap_tbl["Male"] - gap_tbl["Female"]).round(1)
print("Tỷ lệ >50K theo giới, tách theo nhóm giờ làm (%):")
display(gap_tbl)
print(f"- H3: {'ĐƯỢC ỦNG HỘ' if supported else 'KHÔNG ĐƯỢC ỦNG HỘ'} (xem bảng tách theo hôn nhân và giờ làm để đánh giá mức gây nhiễu)")

In [ ]:
t_hr = rate_table("hours_grp")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = [C_HIGH if r > BASE else C_LOW for r in t_hr["rate"]]
bars = axes[0].bar(t_hr.index.astype(str), t_hr["rate_%"], color=colors)
bar_text(axes[0], bars, [f"{r:.1f}%\n(n={n:,})" for r, n in zip(t_hr["rate_%"], t_hr["n"])])
axes[0].axhline(BASE * 100, color="gray", ls="--")
axes[0].set_ylim(0, t_hr["rate_%"].max() + 12)
axes[0].set_title("(a) Tỷ lệ >50K theo nhóm giờ làm/tuần")
axes[0].set_xlabel("Số giờ làm mỗi tuần")
axes[0].set_ylabel("% người thu nhập >50K")
sns.boxplot(data=dfa, x="income", y="hours_per_week", order=["<=50K", ">50K"], palette=PALETTE,
            hue="income", legend=False, ax=axes[1])
axes[1].set_title("(b) Phân phối giờ làm theo nhóm thu nhập")
axes[1].set_xlabel("Nhóm thu nhập")
axes[1].set_ylabel("Giờ làm/tuần")
plt.suptitle("Hình 5. Giờ làm việc và thu nhập", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_fig(fig, "hinh5_gio_lam")
plt.show()

u, p, auc = mw_test("hours_per_week")
rho, _ = spearmanr(dfa["hours_per_week"], dfa["income_bin"])
rate_le39 = dfa.loc[dfa["hours_per_week"] < 40, "income_bin"].mean()
rate_40 = dfa.loc[dfa["hours_per_week"] == 40, "income_bin"].mean()
rate_gt40 = dfa.loc[dfa["hours_per_week"] > 40, "income_bin"].mean()
supported = rate_gt40 > rate_40 > rate_le39 and p < 0.05
ket_qua.append({"Giả thuyết": "H4 Giờ làm nhiều, >50K cao hơn", "Kiểm định": "Mann-Whitney U",
                "p-value": fmt_p(p), "Độ lớn hiệu ứng": f"AUC = {auc:.3f}",
                "Kết luận": "Ủng hộ" if supported else "Không ủng hộ"})

print("SỐ LIỆU CHÍNH (dùng để viết Deep Insight):")
display(t_hr[["n", "rate_%"]])
print(f"- <40 giờ: {rate_le39:.1%} | đúng 40 giờ: {rate_40:.1%} | >40 giờ: {rate_gt40:.1%}")
print(f"- Người làm đúng 40 giờ chiếm {(dfa['hours_per_week'] == 40).mean():.1%} tổng số")
print(f"- Giờ làm trung vị: nhóm >50K = {dfa.loc[dfa['income_bin'] == 1, 'hours_per_week'].median():.0f}, "
      f"nhóm <=50K = {dfa.loc[dfa['income_bin'] == 0, 'hours_per_week'].median():.0f}")
print(f"- Mann-Whitney U: p = {fmt_p(p)}, AUC = {auc:.3f} | Spearman rho = {rho:.3f}")
print(f"- H4: {'ĐƯỢC ỦNG HỘ' if supported else 'KHÔNG ĐƯỢC ỦNG HỘ'}")

In [ ]:
t_mar = rate_table("marital_status").sort_values("rate")
t_occ = rate_table("occupation").sort_values("rate")

fig, axes = plt.subplots(1, 2, figsize=(17, 6.5))
for ax, t, title in [(axes[0], t_mar, "(a) Theo tình trạng hôn nhân"), (axes[1], t_occ, "(b) Theo nghề nghiệp")]:
    colors = [C_HIGH if r > BASE else C_LOW for r in t["rate"]]
    bars = ax.barh(t.index.astype(str), t["rate_%"], color=colors)
    bar_text(ax, bars, [f"{r:.1f}% (n={n:,})" for r, n in zip(t["rate_%"], t["n"])], horizontal=True)
    ax.axvline(BASE * 100, color="gray", ls="--")
    ax.set_xlim(0, t["rate_%"].max() + 22)
    ax.set_title(title)
    ax.set_xlabel("% người thu nhập >50K")
plt.suptitle("Hình 6. Hôn nhân, nghề nghiệp và thu nhập", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_fig(fig, "hinh6_hon_nhan_nghe_nghiep")
plt.show()

chi2_m, p_m, dof_m, v_m = chi2_test(dfa["marital_status"])
chi2_o, p_o, dof_o, v_o = chi2_test(dfa["occupation"])
rate_married = dfa.loc[dfa["is_married"] == 1, "income_bin"].mean()
rate_other = dfa.loc[dfa["is_married"] == 0, "income_bin"].mean()
sup5 = rate_married > rate_other and p_m < 0.05
top2 = set(t_occ["rate"].nlargest(2).index)
sup6 = top2 == {"Exec-managerial", "Prof-specialty"} and p_o < 0.05
ket_qua.append({"Giả thuyết": "H5 Đang kết hôn có >50K cao hơn", "Kiểm định": "Chi-square",
                "p-value": fmt_p(p_m), "Độ lớn hiệu ứng": f"V = {v_m:.3f}",
                "Kết luận": "Ủng hộ" if sup5 else "Không ủng hộ"})
ket_qua.append({"Giả thuyết": "H6 Quản lý/chuyên môn cao nhất", "Kiểm định": "Chi-square",
                "p-value": fmt_p(p_o), "Độ lớn hiệu ứng": f"V = {v_o:.3f}",
                "Kết luận": "Ủng hộ" if sup6 else "Không ủng hộ"})

print("SỐ LIỆU CHÍNH (dùng để viết Deep Insight):")
print("Hôn nhân (giảm dần):")
display(t_mar.sort_values("rate", ascending=False)[["n", "rate_%"]])
print(f"- Civ-spouse: {rate_married:.1%} | các nhóm còn lại gộp: {rate_other:.1%} | gấp {rate_married / rate_other:.2f} lần")
print(f"- Chi-square hôn nhân: p = {fmt_p(p_m)}, Cramér's V = {v_m:.3f}")
print("\nNghề nghiệp (giảm dần):")
display(t_occ.sort_values("rate", ascending=False)[["n", "rate_%"]])
print(f"- Hai nghề cao nhất: {', '.join(t_occ['rate'].nlargest(2).index)} | thấp nhất: {t_occ['rate'].idxmin()} ({t_occ['rate'].min():.1%})")
print(f"- Chi-square nghề nghiệp: p = {fmt_p(p_o)}, Cramér's V = {v_o:.3f}")
print(f"- H5: {'ĐƯỢC ỦNG HỘ' if sup5 else 'KHÔNG ĐƯỢC ỦNG HỘ'} | H6: {'ĐƯỢC ỦNG HỘ' if sup6 else 'KHÔNG ĐƯỢC ỦNG HỘ'}")

In [ ]:
g = dfa.groupby(["age_group", "edu_grp"], observed=True)["income_bin"].agg(["mean", "size"])
rate_p = (g["mean"].unstack() * 100)
n_p = g["size"].unstack()
rate_p = rate_p.where(n_p >= 30)          # che các ô có n < 30

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(rate_p, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.5,
            cbar_kws={"label": "% người thu nhập >50K"}, ax=ax)
ax.set_title("Hình 7. Tỷ lệ >50K (%) theo nhóm tuổi và trình độ học vấn")
ax.set_xlabel("Trình độ học vấn")
ax.set_ylabel("Nhóm tuổi")
plt.tight_layout()
save_fig(fig, "hinh7_heatmap_tuoi_hoc_van")
plt.show()

print("SỐ LIỆU CHÍNH (dùng để viết Deep Insight):")
print("Tỷ lệ >50K (%) theo tuổi x học vấn:")
display(rate_p.round(1))
st = rate_p.stack()
print(f"- Ô cao nhất: tuổi {st.idxmax()[0]} & {st.idxmax()[1]} = {st.max():.1f}%")
print(f"- Ô thấp nhất: tuổi {st.idxmin()[0]} & {st.idxmin()[1]} = {st.min():.1f}%")
print("- So sánh trong cùng nhóm tuổi 45-54: ", end="")
if "45-54" in rate_p.index:
    r = rate_p.loc["45-54"].dropna()
    print(" | ".join(f"{k}: {v:.0f}%" for k, v in r.items()))
else:
    print("không có dữ liệu")
print("- So sánh trong cùng nhóm học vấn Bachelors theo tuổi: ", end="")
if "Bachelors" in rate_p.columns:
    r = rate_p["Bachelors"].dropna()
    print(" | ".join(f"{k}: {v:.0f}%" for k, v in r.items()))

In [ ]:
for c in ["capital_gain", "capital_loss"]:
    if c + "_log" not in dfa.columns:
        dfa[c + "_log"] = np.log1p(dfa[c])

corr_df = pd.DataFrame({
    "income_bin": dfa["income_bin"],
    "age": dfa["age"],
    "education_num": dfa["education_num"],
    "hours_per_week": dfa["hours_per_week"],
    "capital_gain_log": dfa["capital_gain_log"],
    "capital_loss_log": dfa["capital_loss_log"],
    "is_male": dfa["is_male"],
    "is_married": dfa["is_married"],
})
corr = corr_df.corr()
rank = corr["income_bin"].drop("income_bin").sort_values(key=abs, ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={"width_ratios": [1.3, 1]})
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=axes[0])
axes[0].set_title("(a) Ma trận tương quan")
bars = axes[1].barh(rank.index, rank.values, color=[C_HIGH if v > 0 else C_LOW for v in rank.values])
for b, v in zip(bars, rank.values):
    axes[1].text(v + (0.01 if v >= 0 else -0.01), b.get_y() + b.get_height() / 2, f"{v:.2f}",
                 va="center", ha="left" if v >= 0 else "right", fontsize=9)
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_xlim(rank.min() - 0.08, rank.max() + 0.1)
axes[1].set_title("(b) Tương quan với income, xếp theo độ mạnh")
axes[1].set_xlabel("Hệ số tương quan")
plt.suptitle("Hình 8. Các yếu tố liên quan đến thu nhập", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_fig(fig, "hinh8_tuong_quan")
plt.show()

print("SỐ LIỆU CHÍNH (dùng để viết Deep Insight):")
print("Xếp hạng theo |tương quan| với income (mạnh -> yếu):")
for k, v in rank.iloc[::-1].items():
    print(f"  {k:18s} r = {v:+.3f}")
print(f"- Mạnh nhất: {rank.index[-1]} (r = {rank.iloc[-1]:+.3f}); yếu nhất: {rank.index[0]} (r = {rank.iloc[0]:+.3f})")
print(f"- Tương quan giữa is_male và is_married: r = {corr.loc['is_male', 'is_married']:+.3f} (gợi ý yếu tố gây nhiễu)")
print(f"- Tương quan giữa age và is_married: r = {corr.loc['age', 'is_married']:+.3f}")

In [ ]:
bang_tong_ket = pd.DataFrame(ket_qua)
display(bang_tong_ket)

print("Ghi chú đọc kết quả:")
print("- p-value < 0.05: mối liên hệ có ý nghĩa thống kê (với hơn 32.000 dòng, p-value gần như luôn rất nhỏ,")
print("  nên cần xem thêm độ lớn hiệu ứng).")
print("- Cramér's V: <0.1 yếu | 0.1-0.3 trung bình | >0.3 mạnh.")
print("- AUC: 0.5 = không khác biệt; càng xa 0.5 thì khác biệt càng rõ.")
print("\nHOÀN TẤT GIAI ĐOẠN 3.")